<a href="https://colab.research.google.com/github/ratwet/ShadowFox/blob/Preview/ask2_Store_Sales_Analysis/Store_Sales_Profit_Analysis_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Store Sales and Profit Analysis
**Dataset:** Superstore retail dataset 8,399 orders with Sales, Profit, Discount, Category, Sub-Category, Region, Customer Segment, and Order Date fields.

**Goal:** Identify sales trends, top-performing categories, and customer segment profitability to support data-driven business decisions.


## Imports

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Image as IPImage

# Helper to render plotly figures inline in any environment
def show(fig, h=450):
    fig.show()

pd.set_option('display.max_columns', None)
print("Ready.")


Ready.


## Load Data

In [2]:
url = "https://raw.githubusercontent.com/curran/data/gh-pages/superstoreSales/superstoreSales.csv"
df = pd.read_csv(url, encoding='latin1')
print(f"Shape: {df.shape}")
df.head()


Shape: (8399, 21)


,Row ID,Order ID,Order Date,Order Priority,Order Quantity,Sales,Discount,Ship Mode,Profit,Unit Price,Shipping Cost,Customer Name,Province,Region,Customer Segment,Product Category,Product Sub-Category,Product Name,Product Container,Product Base Margin,Ship Date
0,1,3,10/13/2010,Low,6,261.5400,0.04,Regular Air,-213.25,38.94,35.00,Muhammed MacIntyre,Nunavut,Nunavut,Small Business,Office Supplies,Storage & Organization,"Eldon Base for stackable storage shelf, platinum",Large Box,0.80,10/20/2010
1,49,293,10/1/2012,High,49,10123.0200,0.07,Delivery Truck,457.81,208.16,68.02,Barry French,Nunavut,Nunavut,Consumer,Office Supplies,Appliances,"1.7 Cubic Foot Compact ""Cube"" Office Refrigera...",Jumbo Drum,0.58,10/2/2012
2,50,293,10/1/2012,High,27,244.5700,0.01,Regular Air,46.71,8.69,2.99,Barry French,Nunavut,Nunavut,Consumer,Office Supplies,Binders and Binder Accessories,"Cardinal Slant-D¨ Ring Binder, Heavy Gauge Vinyl",Small Box,0.39,10/3/2012
3,80,483,7/10/2011,High,30,4965.7595,0.08,Regular Air,1198.97,195.99,3.99,Clay Rozendal,Nunavut,Nunavut,Corporate,Technology,Telephones and Communication,R380,Small Box,0.58,7/12/2011
4,85,515,8/28/2010,Not Specified,19,394.2700,0.08,Regular Air,30.94,21.78,5.94,Carlos Soltero,Nunavut,Nunavut,Consumer,Office Supplies,Appliances,Holmes HEPA Air Purifier,Medium Box,0.50,8/30/2010


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8399 entries, 0 to 8398
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Row ID                8399 non-null   int64  
 1   Order ID              8399 non-null   int64  
 2   Order Date            8399 non-null   object 
 3   Order Priority        8399 non-null   object 
 4   Order Quantity        8399 non-null   int64  
 5   Sales                 8399 non-null   float64
 6   Discount              8399 non-null   float64
 7   Ship Mode             8399 non-null   object 
 8   Profit                8399 non-null   float64
 9   Unit Price            8399 non-null   float64
 10  Shipping Cost         8399 non-null   float64
 11  Customer Name         8399 non-null   object 
 12  Province              8399 non-null   object 
 13  Region                8399 non-null   object 
 14  Customer Segment      8399 non-null   object 
 15  Product Category     

## Data Preparation

### Convert dates and extract time features

In [4]:
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']   = pd.to_datetime(df['Ship Date'])

df['Year']      = df['Order Date'].dt.year
df['Month']     = df['Order Date'].dt.month
df['YearMonth'] = df['Order Date'].dt.to_period('M').astype(str)

print(df[['Order Date','Year','Month','YearMonth']].head(3))


  Order Date  Year  Month YearMonth
0 2010-10-13  2010     10   2010-10
1 2012-10-01  2012     10   2012-10
2 2012-10-01  2012     10   2012-10


### Check missing values

In [5]:
missing = df.isnull().sum()
print(missing[missing > 0])


Product Base Margin    63
dtype: int64


In [6]:
# Fill missing margin values with category median
df['Product Base Margin'] = df.groupby('Product Category')['Product Base Margin']     .transform(lambda x: x.fillna(x.median()))
print("Missing after fix:", df['Product Base Margin'].isnull().sum())


Missing after fix: 0


### Outlier Detection IQR Method
Using the IQR technique to identify outliers in Sales and Profit columns.

In [7]:
def iqr_bounds(s):
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - 1.5*IQR, Q3 + 1.5*IQR

for col in ['Sales', 'Profit']:
    lo, hi = iqr_bounds(df[col])
    n = ((df[col] < lo) | (df[col] > hi)).sum()
    print(f"{col}: lower={lo:.2f}, upper={hi:.2f}, outliers={n} ({n/len(df)*100:.1f}%)")


Sales: lower=-2205.99, upper=4058.51, outliers=1042 (12.4%)
Profit: lower=-452.41, upper=531.85, outliers=1704 (20.3%)


In [8]:
fig = px.box(df, y=['Sales','Profit'], title="Boxplot: Sales & Profit Outlier Check")
show(fig)


Outliers are kept these represent genuine high-value bulk orders that are important for business analysis, not data errors.

## Sales Analysis

### Monthly Sales Trend

In [9]:
monthly = df.groupby('YearMonth')['Sales'].sum().reset_index()
fig = px.line(monthly, x='YearMonth', y='Sales',
              title='Monthly Sales Over Time', markers=True)
fig.update_xaxes(tickangle=45)
show(fig, h=400)


### Sales & Profit by Category

In [10]:
cat = df.groupby('Product Category')[['Sales','Profit']].sum().reset_index()          .sort_values('Sales', ascending=False)
fig = px.bar(cat, x='Product Category', y=['Sales','Profit'],
             barmode='group', title='Sales and Profit by Category')
show(fig)


### Sub-Category Sales Distribution

In [11]:
subcat = df.groupby('Product Sub-Category')['Sales'].sum().reset_index()
fig = px.treemap(subcat, path=['Product Sub-Category'], values='Sales',
                 title='Sales by Sub-Category')
show(fig, h=500)


## Profit Analysis

### Most and Least Profitable Sub-Categories

In [12]:
subcat_p = df.groupby('Product Sub-Category')['Profit'].sum().reset_index()               .sort_values('Profit')
fig = px.bar(subcat_p, x='Profit', y='Product Sub-Category', orientation='h',
             color='Profit', color_continuous_scale='RdYlGn',
             title='Profit by Sub-Category')
fig.update_layout(yaxis={'categoryorder':'total ascending'})
show(fig, h=600)


### Profit by Customer Segment

In [13]:
seg = df.groupby('Customer Segment')[['Sales','Profit']].sum().reset_index()

fig = px.pie(seg, names='Customer Segment', values='Sales',
             title='Sales Share by Customer Segment', hole=0.4)
show(fig)


In [14]:
fig = px.bar(seg, x='Customer Segment', y='Profit',
             color='Customer Segment', title='Profit by Customer Segment')
show(fig, h=400)


## Operational Insights

### Profit Margin % by Category

In [15]:
cat['Margin %'] = (cat['Profit'] / cat['Sales'] * 100).round(2)
print(cat[['Product Category','Sales','Profit','Margin %']])

fig = px.bar(cat, x='Product Category', y='Margin %',
             color='Margin %', color_continuous_scale='RdYlGn',
             title='Profit Margin % by Category')
show(fig, h=400)


  Product Category        Sales     Profit  Margin %
2       Technology  5984248.182  886313.52     14.81
0        Furniture  5178590.542  117433.03      2.27
1  Office Supplies  3752762.100  518021.43     13.80


### Customer Segment Efficiency

In [16]:
seg['Margin %'] = (seg['Profit'] / seg['Sales'] * 100).round(2)
print(seg.sort_values('Profit', ascending=False))


  Customer Segment         Sales     Profit  Margin %
1        Corporate  5.498905e+06  599746.00     10.91
2      Home Office  3.564764e+06  318354.03      8.93
3   Small Business  2.788321e+06  315708.01     11.32
0         Consumer  3.063611e+06  287959.94      9.40


### Regional Performance

In [17]:
region = df.groupby('Region')[['Sales','Profit']].sum().reset_index()             .sort_values('Sales', ascending=False)
fig = px.bar(region, x='Region', y=['Sales','Profit'], barmode='group',
             title='Sales and Profit by Region')
show(fig, h=400)
